# Biomed GT2KG — GCN avec Sentence-BERT

Premier test d'apprentissage du protocole TDGBench. Les nœuds et relations sont initialisés avec `sentence-transformers/all-MiniLM-L6-v2` (384 dimensions). Le premier lancement construit un cache local ; les lancements suivants le réutilisent.

In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    path
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "pyproject.toml").is_file()
)
DATA_ROOT = PROJECT_ROOT / "benchmark_data"
print("Projet :", PROJECT_ROOT)
print("Données :", DATA_ROOT)

In [ ]:
from tdg_bench import TDGBench

bench = TDGBench(DATA_ROOT)

result = bench.evaluate(
    dataset="biomed",
    graph="GT2KG_tdg",
    model="GCN",
    initial_embedding_model="sentence-transformers/all-MiniLM-L6-v2",
    split_seed=42,
    random_seed=42,
    device="auto",
    max_epochs=200,
    patience=30,
    metrics="all",
    selection_metric="f1_macro",
    class_wise=True,
    verbose=True,
    log_every=5,
)

In [ ]:
from dataclasses import asdict

import pandas as pd

print("Epochs exécutées :", result.epochs_ran)
print("Meilleur epoch :", result.best_epoch)
print("Initial embedding model :", result.initial_embedding_model)
print("Train :", result.train)
print("Validation :", result.validation)
print("Test macro-F1 :", result.test.macro_f1)

display(pd.Series(result.test.values, name="test").to_frame())
display(pd.DataFrame(asdict(row) for row in result.test.per_class))
display(pd.DataFrame(
    result.test.confusion_matrix,
    index=result.test.per_class and [row.label for row in result.test.per_class],
    columns=result.test.per_class and [row.label for row in result.test.per_class],
))